Import packages 

In [1]:
import duckdb
import pyarrow.parquet as pq
import os
import re
import numpy as np
import pandas as pd
from dotenv import load_dotenv

# RAG import packages 
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# Vector Embeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


# import from directory
from llm import llm

load_dotenv("../.env")  # .env is in data_concierge (parent of retrieval)

DATASET_DIR = "dataset"  # or "retrieval/dataset" if running from project root
CSV_OUTPUT_DIR = "csv_files"
KEY_NAME = os.getenv("PARQUET_KEY_NAME")
KEY_VALUE = os.getenv("PARQUET_KEY_VALUE")

/Users/bryan/Desktop/Y3S2/BT4103 Capstone Project/data_concierge/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Exporting all parquet file as a csv file

In [2]:
if False:
    os.makedirs(CSV_OUTPUT_DIR, exist_ok=True)

    con = duckdb.connect()
    con.execute(f"PRAGMA add_parquet_key('{KEY_NAME}', '{KEY_VALUE}');")

    for f in os.listdir(DATASET_DIR):
        if f.endswith(".parquet"):
            parquet_path = os.path.join(DATASET_DIR, f)
            csv_path = os.path.join(CSV_OUTPUT_DIR, f.replace(".parquet", ".csv"))
            con.execute(f"""
                COPY (
                    SELECT * FROM read_parquet(
                        '{parquet_path}',
                        encryption_config = {{footer_key: '{KEY_NAME}'}}
                    )
                ) TO '{csv_path}' (HEADER, DELIMITER ',');
            """)
            print(f"Exported {f} -> {csv_path}")


Parse txt files into column-level chunks

In [3]:
def parse_schema_to_column_chunks(docs_dir: str) -> list[Document]:
    """Parse each COLUMN block into a separate Document with table metadata."""
    chunks = []
    
    for filename in os.listdir(docs_dir):
        if not filename.endswith(".txt"):
            continue
            
        filepath = os.path.join(docs_dir, filename)
        table_name = filename.replace(".txt", "")
        
        with open(filepath, "r") as f:
            content = f.read()
        
        # Extract TABLE line for context
        table_match = re.search(r"TABLE: (\w+) \((\d+) rows\)", content)
        joins_match = re.search(r"JOINS: (.+)", content)
        joins = joins_match.group(1) if joins_match else ""
        
        # Split by COLUMN blocks
        column_blocks = re.split(r"\n(?=COLUMN: )", content)
        
        for block in column_blocks:
            if not block.strip() or block.startswith("TABLE:") or block.startswith("JOINS:"):
                continue
                
            col_match = re.search(r"COLUMN: (\w+(?:\s+\w+)?)", block)
            if not col_match:
                continue
            col_name = col_match.group(1).strip()
            
            # Full column block as chunk content (Data Element, Description, etc.)
            chunk_content = f"TABLE: {table_name}\n{block.strip()}"
            if joins:
                chunk_content += f"\n\nJOINS: {joins}"
            
            doc = Document(
                page_content=chunk_content,
                metadata={
                    "table": table_name,
                    "column": col_name,
                    "source": filename
                }
            )
            chunks.append(doc)
    
    return chunks

Load into vector store and run retrieval

In [4]:
# Parse to column-level chunks
docs_dir = "documents_for_vectordb"
chunks = parse_schema_to_column_chunks(docs_dir)

load_dotenv("../.env")

# OpenRouter Embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Then use embeddings in Chroma
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="schema_columns",
    persist_directory="./chroma_db"
)

# Retrieve relevant columns for a user query
query = "How many patients died and what was the cause?"
retrieved = vectorstore.similarity_search(query, k=5)

for doc in retrieved:
    print(f"Table: {doc.metadata['table']}, Column: {doc.metadata['column']}")
    print(doc.page_content[:200], "...\n")

Table: death, Column: person_id
  Data
TABLE: death
COLUMN: person_id
  Data Element: Patient ID
  Description: Unique deidentified number used to identify a patient
  Datatype: integer

JOINS: Join to person via death.person_id = person.p ...

Table: death, Column: person_id
  Data
TABLE: death
COLUMN: person_id
  Data Element: Patient ID
  Description: Unique deidentified number used to identify a patient
  Datatype: integer

JOINS: Join to person via death.person_id = person.p ...

Table: death, Column: person_id
  Data
TABLE: death
COLUMN: person_id
  Data Element: Patient ID
  Description: Unique deidentified number used to identify a patient
  Datatype: integer

JOINS: Join to person via death.person_id = person.p ...

Table: death, Column: person_id
  Data
TABLE: death
COLUMN: person_id
  Data Element: Patient ID
  Description: Unique deidentified number used to identify a patient
  Datatype: integer

JOINS: Join to person via death.person_id = person.p ...

Table: death, Column

RAG chain for NL-to-SQL

In [5]:
# Prompt for SQL generation
SQL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a SQL expert. Generate DuckDB SQL queries based on the schema context provided.
Use only the tables and columns mentioned in the context. Table names and column names are case-sensitive.
If joining tables, use person_id as the common key."""),
    ("human", """Schema context:
{context}

User question: {question}

Generate the SQL query only. No explanation.""")
])

# RAG chain
def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": lambda x: format_docs(vectorstore.similarity_search(x["question"], k=6)),
     "question": lambda x: x["question"]}
    | SQL_PROMPT
    | llm
    | StrOutputParser()
)

# Use it
sql = rag_chain.invoke({"question": "Count patients by gender"})
print(sql)

```sql
SELECT gender_source_value, COUNT(*) AS patient_count
FROM person
GROUP BY gender_source_value
ORDER BY patient_count DESC;
```
